# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `record_sets` metadata can be accessed to list all available record sets (by `@id`), and each record set lists all its fields (by their `@id`).

In [ ]:
# List all record sets and their fields, referencing them by @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name','N/A')}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    fields = rs.get('fields', [])
    print(f"  Fields: [{', '.join([f.get('@id', '') for f in fields])}]")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s above.

**Example:**
Here, we load *all* record sets as DataFrames, using their `@id` for keys.

In [ ]:
# Collect all record_set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            dataframes[record_set_id] = pd.DataFrame()
        print(f"Loaded: {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Example: show columns from first non-empty DataFrame
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        print(f"First non-empty RecordSet: {main_record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
        break
if main_record_set_id is None:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** All analysis references fields/columns using their `@id` as shown in the metadata above.

In [ ]:
# Pick a main numeric field for EDA -- please update @id below as per actual field from your metadata exploration above!
# Example: let's try to automatically pick a numeric column (float or int)
df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found in the record set for EDA.")
else:
    print(f"Analyzing numeric field: {numeric_field_id}")

    # Filtering example: values above dataset mean
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group-by example: pick a categorical field if present
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        print(f"Grouping filtered records by {group_field_id}...")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show a histogram of the main numeric field, and a box plot grouped by the main group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Box plot grouped by a categorical field if available
    if group_field_id is not None:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'Boxplot of {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using its Croissant schema with `mlcroissant`.
- Record sets and fields are referenced by their `@id`, ensuring precise schema mapping for all analysis.
- Simple filtering, normalization, and grouping were demonstrated on numeric and categorical fields.
- Visualizations give a quick sense of the distribution and group differences in the core quantitative variables.

**Next steps:** Consider domain-specific feature engineering, correlate abundance with experimental metadata, or leverage the record sets for specific protein/peptide discovery tasks.